# 04. A1 = A0 + residual shortcut · 5 fold × 3 seed
**9week** · 계획서: [연구계획서.md](연구계획서.md) · 데이터·분할: [02](02_정제_분할_A0_흐름확인.ipynb)

**연구 질문:** Q1의 일부. 같은 깊이·폭에서 **shortcut만 더하면** 새 사용자의 Macro F1이 좋아지는가?

| 항목 | 내용 |
|---|---|
| A0와 다른 점 | 각 블록의 `[ ]` 출력에 블록 입력을 더함: `ReLU(h)` → `ReLU(h + x)` |
| 같은 점 | 데이터, 분할, 층 수, 채널 64, kernel 5, 학습 설정, head |
| 판정 | A1 − A0을 같은 fold끼리 비교 (계획서 5절: 평균 ≥ +0.5%p 그리고 5개 fold 중 4개 이상) |

채널 수와 길이가 같아서 projection(1×1 Conv)이 필요 없다. 그래서 파라미터 수는 A0와 정확히 같고, **차이는 shortcut 하나뿐**이다.

**실행 방법 (Colab)**
1. 런타임 → 런타임 유형 변경 → **GPU**
2. 위에서부터 모든 셀 실행. 첫 셀에서 Google Drive 연결을 허락한다.
3. 결과 표는 노트북 출력에 남고, 같은 내용이 `9week/runs/` 폴더에 CSV로 저장된다. 이 CSV는 다른 모델과 같은 fold끼리 비교하기 위해 사용한다.

**실행 전 점검 (2026-11-16):** Colab GPU에서 실행합니다. 완료한 fold/seed는 `runs/*_progress.json`에 저장되어 재실행 시 건너뜁니다. 중단된 실행 1회는 다시 학습합니다. 각 실행의 학습 곡선, 검증 예측, 표준화 값, 최적 가중치를 저장합니다. 요약·비교는 15회 완료 후에만 가능합니다. 학습 코드나 구조를 변경한 경우 이전 결과를 별도 보관하고 새 실험으로 시작하세요. 검증 Macro F1은 조기 종료에도 사용하는 개발 지표이며 최종 성능은 보류한 테스트에서 확인합니다.


## 1. 설정
- 데이터, 분할, 학습 설정은 **모든 모델 노트북에서 같다.** 바뀌는 것은 `MODEL_NAME`과 3절의 모델 코드뿐이다.
- 분할 ID는 02에서 확정한 값을 그대로 복사했다.

In [1]:
import os, re, sys, tarfile, time, json, hashlib, platform
import numpy as np
import pandas as pd
from collections import Counter

if 'google.colab' in sys.modules:                       # Colab: Google Drive 연결
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍'
else:                                                   # PC에서 실행할 때
    BASE = r'D:\가천대학교\26-2\1. 기계학습프로그래밍'
ARCHIVE = os.path.join(BASE, 'DATA', 'WISDM', 'WISDM_ar_latest.tar.gz')
RUN_DIR = os.path.join(BASE, '9week', 'runs')
os.makedirs(RUN_DIR, exist_ok=True)

MODEL_NAME = 'A1'
SEEDS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]

ACTIVITIES = ['Walking', 'Jogging', 'Upstairs', 'Downstairs', 'Sitting', 'Standing']
WIN, TRAIN_STRIDE, EVAL_STRIDE = 128, 64, 128
MAX_GAP_NS = 1_000_000_000

TEST_USERS = [6, 21, 24, 27, 29, 34]                    # 최종 평가 전까지 사용하지 않음
GROUPS = [[1, 7, 20, 22, 25, 36],
          [9, 16, 19, 26, 28, 33],
          [2, 3, 4, 14, 30, 31],
          [8, 10, 11, 12, 13, 15],
          [5, 17, 18, 23, 32, 35]]
assert os.path.isfile(ARCHIVE), f'데이터 파일이 없습니다: {ARCHIVE}'
all_users = [u for group in GROUPS for u in group]
assert len(all_users) == len(set(all_users)) == 30
assert not set(all_users) & set(TEST_USERS)
assert set(all_users + TEST_USERS) == set(range(1, 37))
assert set(FOLDS) <= set(range(5)) and set(SEEDS) <= {0, 1, 2}


Dataset SHA-256 verified


## 2. 데이터 읽기와 정제 (02와 같은 규칙)
timestamp 0 삭제 → 완전 중복 삭제 → 같은 시각의 두 번째 값 삭제 → 사용자·행동 변경, 시간 역행, 1초 초과 공백에서 구간 끊기 → 128 이상 구간만 사용. 마지막 줄에 02와 같은 수(구간 424개, 윈도우 7,841개)가 나와야 한다.

In [2]:
with tarfile.open(ARCHIVE) as tar:
    raw = tar.extractfile('WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt').read().decode()

records = []
for token in re.split(r'[;\r\n]+', raw):
    token = token.strip()
    if token == '':
        continue
    fields = token.split(',')
    if len(fields) == 7 and fields[-1].strip() == '':
        fields = fields[:6]
    try:
        if len(fields) != 6:
            raise ValueError
        records.append((int(fields[0]), ACTIVITIES.index(fields[1]), int(fields[2]),
                        float(fields[3]), float(fields[4]), float(fields[5])))
    except ValueError:
        records.append(None)

seen, kept = set(), []
for r in records:
    if r is None:
        kept.append(None)
        continue
    if r[2] == 0 or r in seen:
        continue
    seen.add(r)
    last = kept[-1] if kept else None
    if last is not None and last[:2] == r[:2] and last[2] == r[2]:
        continue
    kept.append(r)

usable, start = [], 0
for i in range(1, len(kept) + 1):
    a = kept[i - 1]
    b = kept[i] if i < len(kept) else None
    if not (a is not None and b is not None and a[:2] == b[:2] and 0 < b[2] - a[2] <= MAX_GAP_NS):
        part = [r for r in kept[start:i] if r is not None]
        if len(part) >= WIN:
            usable.append((part[0][0], part[0][1], np.array([r[3:] for r in part], dtype=np.float32)))
        start = i

print('128 이상 구간:', len(usable), '/ 윈도우(stride 128):',
      sum((len(x) - WIN) // EVAL_STRIDE + 1 for u, a, x in usable))

128 이상 구간: 424 / 윈도우(stride 128): 7841


In [3]:
def make_windows(users, stride, mean, std):
    X, y, who = [], [], []
    for u, a, x in usable:
        if u not in users:
            continue
        x = (x - mean) / std
        for s in range(0, len(x) - WIN + 1, stride):
            X.append(x[s:s + WIN])
            y.append(a)
            who.append(u)
    return np.array(X, dtype=np.float32), np.array(y), np.array(who)

## 3. 모델
기준 모델(A0)과 비교해 바뀐 줄에는 `# [A1 변경]` 주석을 달았다.

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score, recall_score

gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus)
if 'google.colab' in sys.modules and not gpus:
    raise RuntimeError('런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.')

# A1: A0와 같고, 블록 끝에 shortcut 더하기만 추가
def conv_bn(x, filters, relu=True):
    x = layers.Conv1D(filters, 5, padding='same')(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x) if relu else x

def build_model():
    inputs = layers.Input(shape=(WIN, 3))
    x = conv_bn(inputs, 64)                     # stem
    for block in range(2):
        h = conv_bn(x, 64)
        h = conv_bn(h, 64, relu=False)
        x = layers.ReLU()(layers.Add()([h, x]))   # [A1 변경] ReLU(h) → ReLU(h + x): 블록 입력 x를 그대로 더함
        x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(6, activation='softmax')(x)
    return models.Model(inputs, outputs, name=MODEL_NAME)

build_model().summary()

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Model: "A1"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 64)   │      1,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 128, 64)   │          0 │ batch_normal

## 4. 5 fold × 3 seed 학습
fold마다 다음을 반복한다.
1. 검증 = 그 묶음의 6명, 학습 = 나머지 4묶음의 24명.
2. 표준화 통계는 학습 24명의 샘플로만 계산한다.
3. seed 3개로 각각 새 모델을 만들어 학습한다(02와 같은 설정: Adam 0.001, batch 64, 최대 100 epoch, 검증 Macro F1 기준 patience 12, 최고 가중치 복원).
4. 각 학습의 결과를 `rows`에, 사람별 정확도를 `user_rows`에 한 줄씩 기록한다.

`for` 반복문은 이 셀 하나에만 있다(계획서 결정 6).

In [5]:
class ValMacroF1(tf.keras.callbacks.Callback):
    # 매 epoch 끝에 검증 Macro F1을 계산해 logs에 넣는다 → EarlyStopping이 이 값을 본다
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(X_val, verbose=0).argmax(axis=1)
        logs['val_macro_f1'] = f1_score(y_val, pred, labels=range(6), average='macro', zero_division=0)

# 완료한 fold/seed만 재사용한다. 실행 중 끊긴 1회는 처음부터 다시 학습한다.
progress_path = os.path.join(RUN_DIR, f'{MODEL_NAME}_progress.json')
with open(ARCHIVE, 'rb') as f:
    data_sha256 = hashlib.file_digest(f, 'sha256').hexdigest()
protocol = dict(revision='2026-11-16-v1', model=MODEL_NAME, data_sha256=data_sha256,
                groups=GROUPS, test_users=TEST_USERS, window=WIN,
                strides=[TRAIN_STRIDE, EVAL_STRIDE], max_gap_ns=MAX_GAP_NS,
                optimizer='Adam', lr=0.001, batch_size=64, max_epochs=100,
                monitor='val_macro_f1', patience=12,
                tensorflow=tf.__version__, numpy=np.__version__)
if os.path.exists(progress_path):
    with open(progress_path, encoding='utf-8') as f:
        saved = json.load(f)
    if saved['protocol'] != protocol:
        raise ValueError('저장된 실험 조건과 다릅니다. 기존 결과를 별도 보관한 뒤 실행하세요.')
    rows, user_rows = saved['rows'], saved['users']
else:
    rows, user_rows = [], []
completed = {(r['fold'], r['seed']) for r in rows}
assert len(completed) == len(rows)
assert completed <= {(f, s) for f in range(5) for s in range(3)}
print('이미 완료한 학습:', sorted(completed))
print('실행 환경:', platform.platform(), 'TensorFlow', tf.__version__)

for fold in FOLDS:
    val_users = GROUPS[fold]
    train_users = [u for i, g in enumerate(GROUPS) if i != fold for u in g]
    train_samples = np.concatenate([x for u, a, x in usable if u in train_users])
    mean, std = train_samples.mean(axis=0), train_samples.std(axis=0)
    assert np.isfinite(std).all() and (std > 0).all()
    X_train, y_train, _ = make_windows(train_users, TRAIN_STRIDE, mean, std)
    X_val, y_val, u_val = make_windows(val_users, EVAL_STRIDE, mean, std)

    assert set(np.unique(y_train)) == set(range(6))
    assert set(np.unique(y_val)) == set(range(6))
    for seed in SEEDS:
        if (fold, seed) in completed:
            continue
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        model = build_model()
        model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        early = tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=12,
                                                 restore_best_weights=True)
        t0 = time.time()
        history = model.fit(X_train, y_train, epochs=100, batch_size=64,
                            validation_data=(X_val, y_val),
                            callbacks=[ValMacroF1(), early], verbose=2)
        seconds = time.time() - t0

        pred = model.predict(X_val, verbose=0).argmax(axis=1)
        recalls = recall_score(y_val, pred, labels=range(6), average=None, zero_division=0)
        row = dict(model=MODEL_NAME, fold=fold, seed=seed,
                   accuracy=np.mean(pred == y_val), macro_f1=f1_score(y_val, pred, labels=range(6), average='macro', zero_division=0),
                   best_epoch=int(np.argmax(history.history['val_macro_f1'])) + 1,
                   epochs=len(history.history['loss']), train_sec=round(seconds, 1),
                   params=model.count_params())
        row.update({f'recall_{n}': r for n, r in zip(ACTIVITIES, recalls)})
        rows.append(row)
        for u in val_users:
            m = u_val == u
            user_rows.append(dict(model=MODEL_NAME, fold=fold, seed=seed, user=u,
                                  windows=int(m.sum()), accuracy=np.mean(pred[m] == y_val[m])))
        # 각 실행의 곡선·예측·표준화 값·최적 가중치를 보존한다.
        prefix = os.path.join(RUN_DIR, f'{MODEL_NAME}_fold{fold}_seed{seed}')
        pd.DataFrame(history.history).to_csv(prefix + '_history.csv', index=False)
        np.savez_compressed(prefix + '_predictions.npz', y_true=y_val, y_pred=pred,
                            users=u_val, mean=mean, std=std)
        model.save_weights(prefix + '.weights.h5')
        with open(progress_path + '.tmp', 'w', encoding='utf-8') as f:
            json.dump(dict(protocol=protocol, rows=rows, users=user_rows), f, ensure_ascii=False)
        os.replace(progress_path + '.tmp', progress_path)
        completed.add((fold, seed))
        pd.DataFrame(rows).to_csv(os.path.join(RUN_DIR, f'{MODEL_NAME}_runs.csv'), index=False)
        pd.DataFrame(user_rows).to_csv(os.path.join(RUN_DIR, f'{MODEL_NAME}_users.csv'), index=False)
        print(f"fold {fold} seed {seed}: acc {row['accuracy']:.4f}  macroF1 {row['macro_f1']:.4f}  "
              f"best epoch {row['best_epoch']:3d}/{row['epochs']:3d}  {seconds:5.0f}s")

runs = pd.DataFrame(rows)
users = pd.DataFrame(user_rows)
runs.to_csv(os.path.join(RUN_DIR, f'{MODEL_NAME}_runs.csv'), index=False)
users.to_csv(os.path.join(RUN_DIR, f'{MODEL_NAME}_users.csv'), index=False)

이미 완료한 학습: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2), (3, 0), (3, 1), (3, 2), (4, 0), (4, 1)]
실행 환경: Linux-6.6.159+-x86_64-with-glibc2.39 TensorFlow 2.21.0
Epoch 1/100
162/162 - 24s - 149ms/step - accuracy: 0.8343 - loss: 0.5052 - val_accuracy: 0.3607 - val_loss: 1.7306 - val_macro_f1: 0.4504
Epoch 2/100
162/162 - 1s - 6ms/step - accuracy: 0.9588 - loss: 0.1340 - val_accuracy: 0.8129 - val_loss: 0.5793 - val_macro_f1: 0.8167
Epoch 3/100
162/162 - 1s - 6ms/step - accuracy: 0.9776 - loss: 0.0747 - val_accuracy: 0.8247 - val_loss: 0.4965 - val_macro_f1: 0.8035
Epoch 4/100
162/162 - 1s - 6ms/step - accuracy: 0.9818 - loss: 0.0626 - val_accuracy: 0.8713 - val_loss: 0.4590 - val_macro_f1: 0.8438
Epoch 5/100
162/162 - 1s - 6ms/step - accuracy: 0.9888 - loss: 0.0409 - val_accuracy: 0.8730 - val_loss: 0.5415 - val_macro_f1: 0.8405
Epoch 6/100
162/162 - 1s - 6ms/step - accuracy: 0.9910 - loss: 0.0337 - val_accuracy: 0.8916 - val_loss: 0.4711 - val_macro_f1: 0.8683
E

## 5. 결과 요약
- **fold별:** seed 3개 평균. 다른 모델과 같은 fold끼리 비교할 때 이 값을 쓴다.
- **전체:** fold별 값의 평균과 표준편차. 표준편차가 크면 사람 조합에 따라 점수가 크게 달라진다는 뜻이다.
- **seed 표준편차:** 같은 fold 안에서 seed만 바꿨을 때의 흔들림.

In [6]:
# 일부 fold/seed 결과를 정식 5-fold 결과로 해석하지 않는다.
expected = {(f, s) for f in range(5) for s in range(3)}
assert len(runs) == 15 and set(zip(runs['fold'], runs['seed'])) == expected, '15회 학습 완료 후 요약하세요.'
assert len(users) == 90 and not users.duplicated(['fold', 'seed', 'user']).any()
per_fold = runs.groupby('fold')[['accuracy', 'macro_f1']].mean()
per_fold['macro_f1_seed_std'] = runs.groupby('fold')['macro_f1'].std()
print(per_fold.round(4), '\n')
print(f"{MODEL_NAME}: Macro F1 {per_fold['macro_f1'].mean():.4f} ± {per_fold['macro_f1'].std():.4f} (fold 간)"
      f" / Accuracy {per_fold['accuracy'].mean():.4f} ± {per_fold['accuracy'].std():.4f}")
print(f"파라미터 {runs['params'].iloc[0]:,} / 평균 최고 epoch {runs['best_epoch'].mean():.1f}"
      f" / 평균 학습시간 {runs['train_sec'].mean():.0f}초")
print(f"Macro F1 ≥ 0.90인 fold: {(per_fold['macro_f1'] >= 0.90).sum()}/5")

print('\n행동별 recall (전체 평균)')
print(runs[[f'recall_{n}' for n in ACTIVITIES]].mean().round(3).to_string())

print('\n사람별 정확도 (seed 평균, 낮은 순 8명)')
print(users.groupby(['fold', 'user'])['accuracy'].mean().sort_values().head(8).round(3).to_string())
print('사람별 동일 비중 평균 정확도:', round(users.groupby('user')['accuracy'].mean().mean(), 4))


      accuracy  macro_f1  macro_f1_seed_std
fold                                       
0       0.9147    0.9174             0.0080
1       0.8825    0.8703             0.0188
2       0.8381    0.8179             0.0147
3       0.9430    0.9360             0.0083
4       0.9125    0.8987             0.0108 

A1: Macro F1 0.8881 ± 0.0461 (fold 간) / Accuracy 0.8982 ± 0.0398
파라미터 84,870 / 평균 최고 epoch 18.7 / 평균 학습시간 46초
Macro F1 ≥ 0.90인 fold: 2/5

행동별 recall (전체 평균)
recall_Walking       0.892
recall_Jogging       0.944
recall_Upstairs      0.794
recall_Downstairs    0.809
recall_Sitting       0.904
recall_Standing      0.997

사람별 정확도 (seed 평균, 낮은 순 8명)
fold  user
2     30      0.311
0     25      0.408
1     9       0.471
4     17      0.556
2     2       0.744
3     12      0.783
1     19      0.826
2     4       0.841
사람별 동일 비중 평균 정확도: 0.8714


## 6. 비교: A0와 같은 fold끼리
비교 대상 노트북을 먼저 실행해 `runs/` 폴더에 CSV가 있어야 한다.

**① fold 판정** (계획서 5절): 같은 fold끼리 Macro F1 차이(seed 3개 평균)를 구한다.
- 개선 = 평균 ≥ +0.5%p 그리고 5개 fold 중 4개 이상에서 개선
- 악화 = 평균 ≤ −0.5%p 그리고 5개 fold 중 4개 이상에서 악화
- 그 외 = 구별 불가

**② 사람별 변화** (개발 30명): 개선이 여러 사람에게서 나왔는지, 한두 명에게서만 나왔는지 확인한다.
- 정확도가 1%p 넘게 오른 사람 수와 내린 사람 수
- 기준 모델에서 가장 낮았던 사람의 변화
- 변화가 가장 큰 3명

In [7]:
COMPARE_WITH = ['A0']
my_users = users.groupby('user')['accuracy'].mean()
for other in COMPARE_WITH:
    other_runs = pd.read_csv(os.path.join(RUN_DIR, f'{other}_runs.csv'))
    assert len(other_runs) == 15 and set(zip(other_runs['fold'], other_runs['seed'])) == expected, f'{other}: 15회 결과가 필요합니다.'
    other_users = pd.read_csv(os.path.join(RUN_DIR, f'{other}_users.csv')).groupby('user')['accuracy'].mean()
    with open(os.path.join(RUN_DIR, f'{other}_progress.json'), encoding='utf-8') as f:
        other_protocol = json.load(f)['protocol']
    assert {k: v for k, v in protocol.items() if k != 'model'} == {k: v for k, v in other_protocol.items() if k != 'model'}, '모델 간 데이터·분할·학습 조건·버전이 다릅니다.'
    assert set(other_users.index) == set(my_users.index), '비교할 사용자 집합이 다릅니다.'
    base = other_runs.groupby('fold')['macro_f1'].mean()
    diff = (per_fold['macro_f1'] - base) * 100                       # %p
    wins, losses = (diff > 0).sum(), (diff < 0).sum()
    if diff.mean() >= 0.5 and wins >= 4:
        verdict = '개선'
    elif diff.mean() <= -0.5 and losses >= 4:
        verdict = '악화'
    else:
        verdict = '구별 불가'
    print(f'========== {MODEL_NAME} vs {other} ==========')
    print(pd.DataFrame({other: base.round(4), MODEL_NAME: per_fold['macro_f1'].round(4), '차이(%p)': diff.round(2)}))
    print(f'① 평균 차이 {diff.mean():+.2f}%p, 개선 fold {wins}/5 → 판정: {verdict}')

    d = (my_users - other_users) * 100
    worst = other_users.idxmin()
    print(f'② 사람별: 1%p 넘게 개선 {(d > 1).sum()}명 / 1%p 넘게 악화 {(d < -1).sum()}명 / 나머지 {((d >= -1) & (d <= 1)).sum()}명')
    print(f'   {other}에서 가장 낮았던 user {worst}: {other_users[worst]:.3f} → {my_users[worst]:.3f}')
    print('   변화가 가장 큰 3명(%p):', d.reindex(d.abs().sort_values(ascending=False).index).head(3).round(1).to_dict())
    print(f"   파라미터: {other} {other_runs['params'].iloc[0]:,} → {MODEL_NAME} {runs['params'].iloc[0]:,}\n")

========== A1 vs A0 ==========
          A0      A1  차이(%p)
fold                        
0     0.9197  0.9174   -0.23
1     0.8857  0.8703   -1.54
2     0.8117  0.8179    0.61
3     0.9388  0.9360   -0.29
4     0.9085  0.8987   -0.98
① 평균 차이 -0.49%p, 개선 fold 1/5 → 판정: 구별 불가
② 사람별: 1%p 넘게 개선 5명 / 1%p 넘게 악화 6명 / 나머지 19명
   A0에서 가장 낮았던 user 30: 0.227 → 0.311
   변화가 가장 큰 3명(%p): {17: -11.2, 30: 8.5, 19: -6.3}
   파라미터: A0 84,870 → A1 84,870



## 결과 해석

Macro F1 0.8881 ± 0.0461.

A0 대비 평균 -0.49%p, 1/5 fold 상승: **사전 기준으로 구별 불가**. 사용자별 평균 정확도는 12명 상승, 16명 하락, 2명 동일하다.

[별도 연구보고서](reports/04_A1_연구보고서.md)
